# Feature engineering inspection: unexecuted input snapshot

## Goal
Inspect the optional `Time + log1p(Amount) + PCA(V1..V28, 10 components)` experiment. The input contract remains 30 raw features; the transformed model sees 12 features. V1..V28 in the public credit-card benchmark are already anonymized PCA-derived features. Reducing them again may remove useful information. These plots do not establish improved fraud detection, calibration, or production readiness.

This copy has no saved results. It checks a declared CSV hash against the bytes it parses; a matching hash does not establish the source or data kind. The original notebook preserves the historical outputs. Synthetic runs check mechanics only.

## Setup
Run the command below from the repository root. Interactive use also supports the notebooks/ directory. Use the repository Python environment and install notebook tools separately if needed:

```sh
python -m pip install nbformat==5.10.4 nbclient==0.10.2 ipykernel==6.29.5 nbconvert==7.16.6
run_dir=$(mktemp -d)
DATA_CSV=/absolute/path/to/creditcard.csv python -m jupyter nbconvert --execute --to notebook --output feature-engineering-verified.ipynb --output-dir "$run_dir" notebooks/feature_engineering_verified.ipynb
```

For interactive use, set `DATA_CSV` in the kernel environment before starting Jupyter. `DATA_KIND` is a declared source label, not independent validation. If absent, a sibling `.metadata.json` supplies it; otherwise plots say `unspecified`.

### Assumptions
- The CSV has the canonical `Time`, `V1`..`V28`, `Amount`, and binary `Class` columns.
- `Amount` is finite and nonnegative. `log1p` supports zero without shifting the data artificially.
- The deterministic stratified split is exploratory. PCA is fitted only on the training partition; a chronological/future holdout is needed for temporal generalization claims.

In [ ]:
get_ipython().run_line_magic("matplotlib", "inline")

import hashlib
import json
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

project_root = Path.cwd().resolve()
if not (project_root / "training").is_dir():
    project_root = project_root.parent
if not (project_root / "training").is_dir():
    raise RuntimeError("Run this notebook from the repository root or notebooks/.")
sys.path.insert(0, str(project_root))

from training.features import FraudFeatureEngineer
from training.preprocess import validate_dataset

random_state = 42
n_components = 10


## Load and identify the data
The reported hash is computed from the bytes used to parse the CSV. Declared sidecar hashes beside both the supplied and resolved paths are checked when present. If both have metadata, fields from the supplied path take precedence; source and source_url are treated as aliases for the source label. The declared data kind is not proof of origin.

In [ ]:
import io
from training.dataset_provenance import read_dataset_sidecar

data_csv = os.getenv("DATA_CSV")
if not data_csv:
    raise RuntimeError("Set DATA_CSV to the labelled fraud CSV. No dataset was substituted.")
source = Path(data_csv).expanduser()
if not source.is_absolute():
    source = project_root / source
display_source = source.resolve(strict=True)
source_bytes = display_source.read_bytes()
target_provenance = read_dataset_sidecar(display_source, source_bytes)
supplied_provenance = read_dataset_sidecar(source, source_bytes) if source != display_source else {}
provenance = {**target_provenance, **supplied_provenance}
dataset_source = (supplied_provenance.get("source_url") or supplied_provenance.get("source")
                  or target_provenance.get("source_url") or target_provenance.get("source"))
data_kind = os.getenv("DATA_KIND") or provenance.get("data_kind", "unspecified")
dataset_sha256 = hashlib.sha256(source_bytes).hexdigest()
compression = pd.io.common.infer_compression(display_source, "infer")
frame = pd.read_csv(io.BytesIO(source_bytes), compression=compression)
raw_features, labels = validate_dataset(frame)
if (raw_features["Amount"] < 0).any():
    raise ValueError("Amount must be nonnegative for log1p.")
print(json.dumps({"source_url": dataset_source,
                  "source": str(display_source.relative_to(project_root)) if display_source.is_relative_to(project_root) else display_source.name, "dataset_sha256": dataset_sha256,
                  "declared_data_kind": data_kind, "rows": len(frame),
                  "raw_feature_count": raw_features.shape[1]}, indent=2))


## Inspect class balance
Use the observed class counts and fraud prevalence. The source label stays visible, but it does not prove the data origin.

In [ ]:
counts = labels.value_counts().reindex([0, 1], fill_value=0)
fig, axis = plt.subplots(figsize=(6, 4))
axis.bar(["Class 0", "Class 1"], counts.to_numpy(), color=["#185a79", "#a32932"])
axis.set(ylabel="Rows", title=f"Observed class balance ({data_kind})")
for index, count in enumerate(counts):
    axis.annotate(f"{count:,}", (index, count), xytext=(0, 5), textcoords="offset points", ha="center")
axis.set_ylim(0, counts.max() * 1.15)
fig.tight_layout()
plt.show()
print(f"Fraud prevalence in supplied data: {labels.mean():.6%}")


## Inspect the amount transformation
Both histograms use the same observed rows. They show the effect of `log1p`; appearance alone is not a model-quality result.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(raw_features["Amount"], bins=50, color="#185a79")
axes[0].set(xlabel="Raw Amount", ylabel="Rows", title="Raw transaction amount")
axes[1].hist(np.log1p(raw_features["Amount"]), bins=50, color="#187d70")
axes[1].set(xlabel="log1p(Amount)", ylabel="Rows", title="Transformed transaction amount")
fig.suptitle(f"Amount distributions ({data_kind}; {len(frame):,} rows)")
fig.tight_layout()
plt.show()


## Fit PCA on the training partition
The held-out rows are transformed with the fitted training PCA. They do not influence its means or components. In model selection, put this transformer inside every cross-validation training fold, before the scaler and SMOTE.

In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    raw_features, labels, test_size=0.2, random_state=random_state, stratify=labels,
)
feature_engineer = FraudFeatureEngineer(n_components=n_components, random_state=random_state)
train_transformed = feature_engineer.fit_transform(X_train)
holdout_transformed = feature_engineer.transform(X_holdout)
variance = feature_engineer.pca_.explained_variance_ratio_
if not np.isfinite(variance).all():
    raise ValueError("PCA variance is undefined for this training dataset; inspect constant features.")
print(f"Training rows: {len(X_train):,}; held-out rows: {len(X_holdout):,}")
print("Transformed columns:", list(feature_engineer.get_feature_names_out()))


## Inspect retained training variance
Explained variance measures representation of V1..V28, not fraud discrimination. A component with low overall variance can still matter for the rare class.

In [ ]:
component_numbers = np.arange(1, n_components + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(component_numbers, variance, color="#185a79")
axes[0].set(xlabel="PCA component", ylabel="Explained variance ratio", title="Per-component training variance")
axes[1].plot(component_numbers, np.cumsum(variance), marker="o", color="#187d70")
axes[1].set(xlabel="PCA components retained", ylabel="Cumulative explained variance ratio", ylim=(0, 1.02), title="Retained training variance")
for axis in axes:
    axis.set_xticks(component_numbers)
fig.suptitle(f"Additional PCA over V1..V28 ({data_kind}; training partition only)")
fig.tight_layout()
plt.show()
print(f"Training variance retained by {n_components} components: {variance.sum():.6%}")


## Checks
These checks verify the transformation contract, not real-world model performance.

In [ ]:
assert raw_features.shape[1] == 30
assert train_transformed.shape == (len(X_train), 12)
assert holdout_transformed.shape == (len(X_holdout), 12)
assert np.isfinite(train_transformed.to_numpy()).all()
assert np.isfinite(holdout_transformed.to_numpy()).all()
np.testing.assert_allclose(feature_engineer.pca_.mean_, X_train[[f"V{i}" for i in range(1, 29)]].mean())
pd.testing.assert_frame_equal(holdout_transformed, feature_engineer.transform(X_holdout))
print("Raw 30-feature input preserved; fitted transformer produces 12 finite features.")


## Next steps
Keep the experiment opt-in until the baseline and engineered pipeline are compared using the same untouched labelled holdout and explicit operating threshold. Report ROC AUC, average precision, recall, precision, and false-positive burden. Do not interpret PCA variance or cleaner-looking histograms as a measured quality improvement. Synthetic runs validate this notebook's mechanics only.